# OpenBridge — 04 · Análisis Exploratorio de Datos (EDA) e Interfaz para la IA

**Módulo:** Data Engineering / Contratos de IA  
**Objetivo:** Importar los datos cargados en PostgreSQL desde la fase de ingesta, realizar el Análisis Exploratorio de Datos (EDA) enfocado en las necesidades del LLM/RAG y construir la vista de interfaz `v_repos_for_ai`.  
**Entrada:** Tablas `catalog_repositories` y `catalog_readmes` en PostgreSQL.  
**Salida:** Vista relacional `v_repos_for_ai` para consumo exclusivo del equipo de IA.

In [ ]:
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

# 1. Cargar variables de entorno
load_dotenv()

DB_USER = os.getenv("DB_USER", "tu_usuario")
DB_PASS = os.getenv("DB_PASSWORD", "tu_contraseña")
DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = os.getenv("DB_PORT", "5432")
DB_NAME = os.getenv("DB_NAME", "openbridge_db")

# 2. Conectar a PostgreSQL
from sqlalchemy.engine import URL

engine = create_engine(
    URL.create(
        "postgresql+psycopg",
        username=DB_USER,
        password=DB_PASS,
        host=DB_HOST,
        port=int(DB_PORT),
        database=DB_NAME
    )
)

with engine.begin() as conn:
    # A. Renombrar 'id' a 'github_id' si existe con el nombre antiguo
    conn.execute(text("""
        DO $$ 
        BEGIN 
            IF EXISTS (
                SELECT 1 FROM information_schema.columns 
                WHERE table_name='catalog_repositories' AND column_name='id'
            ) THEN
                ALTER TABLE catalog_repositories RENAME COLUMN id TO github_id;
            END IF;
        END $$;
    """))

    # B. Asignar PRIMARY KEY a github_id en catalog_repositories si no la tiene
    conn.execute(text("""
        DO $$ 
        BEGIN 
            IF NOT EXISTS (
                SELECT 1 FROM information_schema.table_constraints 
                WHERE table_name = 'catalog_repositories' AND constraint_type = 'PRIMARY KEY'
            ) THEN
                ALTER TABLE catalog_repositories ADD PRIMARY KEY (github_id);
            END IF;
        END $$;
    """))

    # C. Crear la tabla catalog_readmes con la llave foránea corregida
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS catalog_readmes (
            github_id BIGINT PRIMARY KEY REFERENCES catalog_repositories(github_id) ON DELETE CASCADE,
            readme_text TEXT,
            updated_at TIMESTAMPTZ DEFAULT CURRENT_TIMESTAMP
        );
    """))

print("[OK] Clave primaria agregada a 'catalog_repositories' y tabla 'catalog_readmes' creada con éxito.")

# 3. Cargar a Pandas
df_repos = pd.read_sql("SELECT * FROM catalog_repositories;", con=engine)
df_readmes = pd.read_sql("SELECT * FROM catalog_readmes;", con=engine)

print(f"[*] Importación exitosa desde PostgreSQL:")
print(f"    -> Repositorios: {df_repos.shape[0]} filas x {df_repos.shape[1]} columnas")
print(f"    -> READMEs:      {df_readmes.shape[0]} filas x {df_readmes.shape[1]} columnas")

# OpenBridge — 04 · Ingesta de READMEs, Hash de Contenido y Vista para la IA

**Módulo:** Data Engineering / Contratos e Interfaz de IA  
**Autor:** Jeison Zapata Pinzón (Data Lead)  
**Objetivo:** 
1. Garantizar la integridad estructural del esquema en PostgreSQL (`github_id` como `PRIMARY KEY`).
2. Extraer los textos `README.md` reales de cada repositorio mediante la API de GitHub.
3. Generar el `content_hash` (SHA-256) combinando el texto del README y la marca de tiempo `pushed_at`.
4. Poblado de la tabla `catalog_readmes` y actualización de `catalog_repositories`.
5. Despliegue de la vista relacional `v_repos_for_ai` para consumo exclusivo del equipo de IA.

In [ ]:
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

# 1. Cargar variables de entorno
load_dotenv()

DB_USER = os.getenv("DB_USER", "tu_usuario")
DB_PASS = os.getenv("DB_PASSWORD", "tu_contraseña")
DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = os.getenv("DB_PORT", "5432")
DB_NAME = os.getenv("DB_NAME", "openbridge_db")

# 2. Conectar a PostgreSQL con psycopg v3
from sqlalchemy.engine import URL

engine = create_engine(
    URL.create(
        "postgresql+psycopg",
        username=DB_USER,
        password=DB_PASS,
        host=DB_HOST,
        port=int(DB_PORT),
        database=DB_NAME
    )
)

# 3. Alineación defensiva del esquema relacional
with engine.begin() as conn:
    # A. Renombrar 'id' a 'github_id' si existe con el nombre antiguo
    conn.execute(text("""
        DO $$ BEGIN IF EXISTS ( SELECT 1 FROM information_schema.columns WHERE table_name='catalog_repositories' AND column_name='id' ) THEN ALTER TABLE catalog_repositories RENAME COLUMN id TO github_id; END IF; END $$;
    """))

    # B. Garantizar PRIMARY KEY en github_id de catalog_repositories
    conn.execute(text("""
        DO $$ BEGIN IF NOT EXISTS ( SELECT 1 FROM information_schema.table_constraints WHERE table_name = 'catalog_repositories' AND constraint_type = 'PRIMARY KEY' ) THEN ALTER TABLE catalog_repositories ADD PRIMARY KEY (github_id); END IF; END $$;
    """))

    # C. Crear la tabla catalog_readmes vinculada por Foreign Key
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS catalog_readmes (
            github_id BIGINT PRIMARY KEY REFERENCES catalog_repositories(github_id) ON DELETE CASCADE,
            readme_text TEXT,
            updated_at TIMESTAMPTZ DEFAULT CURRENT_TIMESTAMP
        );
    """))

print("[OK] Esquema preparado: 'catalog_repositories' posee PRIMARY KEY en 'github_id' y 'catalog_readmes' está creada.")

## 1. Extracción de READMEs y Generación de `content_hash`

Según el **Contrato de IA (T05)**:
- La IA necesita el campo `readme_text` para clasificar las categorías, casos de uso e ideas de integración.
- Para evitar reprocesamientos costosos en el LLM, se calcula un `content_hash` único mediante la fórmula:
  $$\text{content\_hash} = \text{SHA256}(\text{readme\_text} + \text{pushed\_at})$$
- La IA comparará este hash contra su histórico para procesar **únicamente repositorios nuevos o que hayan cambiado**.

In [ ]:
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

# 1. Cargar variables de entorno
load_dotenv()

DB_USER = os.getenv("DB_USER", "tu_usuario")
DB_PASS = os.getenv("DB_PASSWORD", "tu_contraseña")
DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = os.getenv("DB_PORT", "5432")
DB_NAME = os.getenv("DB_NAME", "openbridge_db")

# 2. Conectar a PostgreSQL
from sqlalchemy.engine import URL

engine = create_engine(
    URL.create(
        "postgresql+psycopg",
        username=DB_USER,
        password=DB_PASS,
        host=DB_HOST,
        port=int(DB_PORT),
        database=DB_NAME
    )
)

# 3. Ajuste defensivo de esquema: agregar columnas y restricciones faltantes
with engine.begin() as conn:
    # A. Renombrar 'id' a 'github_id' si existe con el nombre antiguo
    conn.execute(text("""
        DO $$ 
        BEGIN 
            IF EXISTS (
                SELECT 1 FROM information_schema.columns 
                WHERE table_name='catalog_repositories' AND column_name='id'
            ) THEN
                ALTER TABLE catalog_repositories RENAME COLUMN id TO github_id;
            END IF;
        END $$;
    """))

    # B. Asignar PRIMARY KEY a github_id en catalog_repositories
    conn.execute(text("""
        DO $$ 
        BEGIN 
            IF NOT EXISTS (
                SELECT 1 FROM information_schema.table_constraints 
                WHERE table_name = 'catalog_repositories' AND constraint_type = 'PRIMARY KEY'
            ) THEN
                ALTER TABLE catalog_repositories ADD PRIMARY KEY (github_id);
            END IF;
        END $$;
    """))

    # C. Agregar la columna content_hash si no existe en catalog_repositories
    conn.execute(text("""
        ALTER TABLE catalog_repositories 
        ADD COLUMN IF NOT EXISTS content_hash VARCHAR(64);
    """))

    # D. Crear la tabla catalog_readmes
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS catalog_readmes (
            github_id BIGINT PRIMARY KEY REFERENCES catalog_repositories(github_id) ON DELETE CASCADE,
            readme_text TEXT,
            updated_at TIMESTAMPTZ DEFAULT CURRENT_TIMESTAMP
        );
    """))

print("[OK] Columna 'content_hash' agregada y esquema listo para el UPDATE.")

In [ ]:
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

# 1. Cargar variables de entorno y conexión
load_dotenv()

DB_USER = os.getenv("DB_USER", "tu_usuario")
DB_PASS = os.getenv("DB_PASSWORD", "tu_contraseña")
DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = os.getenv("DB_PORT", "5432")
DB_NAME = os.getenv("DB_NAME", "openbridge_db")

from sqlalchemy.engine import URL

engine = create_engine(
    URL.create(
        "postgresql+psycopg",
        username=DB_USER,
        password=DB_PASS,
        host=DB_HOST,
        port=int(DB_PORT),
        database=DB_NAME
    )
)

# 2. Renombrar columnas con puntos a guiones bajos en catalog_repositories
with engine.begin() as conn:
    conn.execute(text("""
        DO $$ 
        BEGIN 
            -- Renombrar license.spdx_id -> license_spdx_id
            IF EXISTS (
                SELECT 1 FROM information_schema.columns 
                WHERE table_name='catalog_repositories' AND column_name='license.spdx_id'
            ) THEN
                ALTER TABLE catalog_repositories RENAME COLUMN "license.spdx_id" TO license_spdx_id;
            END IF;

            -- Renombrar owner.id -> owner_id
            IF EXISTS (
                SELECT 1 FROM information_schema.columns 
                WHERE table_name='catalog_repositories' AND column_name='owner.id'
            ) THEN
                ALTER TABLE catalog_repositories RENAME COLUMN "owner.id" TO owner_id;
            END IF;

            -- Renombrar owner.login -> owner_login
            IF EXISTS (
                SELECT 1 FROM information_schema.columns 
                WHERE table_name='catalog_repositories' AND column_name='owner.login'
            ) THEN
                ALTER TABLE catalog_repositories RENAME COLUMN "owner.login" TO owner_login;
            END IF;
        END $$;
    """))

print("[OK] Columnas con puntos corregidas a formato PostgreSQL ('license_spdx_id').")

# 3. Crear la vista v_repos_for_ai adaptable al nombre de columna existente
ddl_view = """
CREATE OR REPLACE VIEW v_repos_for_ai AS
SELECT 
    r.github_id,
    r.full_name,
    r.name,
    r.description,
    r.homepage,
    r.language,
    r.topics,
    r.stargazers_count,
    r.forks_count,
    r.pushed_at,
    r.archived,
    r.license_spdx_id,
    r.content_hash,
    m.readme_text
FROM catalog_repositories r
LEFT JOIN catalog_readmes m ON r.github_id = m.github_id;
"""

with engine.begin() as conn:
    conn.execute(text(ddl_view))
    print("[OK] Vista 'v_repos_for_ai' desplegada correctamente.")

In [ ]:
df_check = pd.read_sql("""
    SELECT 
        github_id, 
        full_name, 
        language, 
        license_spdx_id,
        content_hash, 
        LENGTH(readme_text) AS caracteres_readme 
    FROM v_repos_for_ai;
""", con=engine)

print("=== CONTRATO CUMPLIDO: VISTA LISTA PARA EL MÓDULO DE IA ===")
display(df_check)

Entrega de Interfaz y Contrato de Datos
De: Jeison Zapata Pinzón (Ingesta y Catálogo)

Para: Equipo de Inteligencia Artificial (AI Lead / AI Engineer)

Asunto: Disponibilidad de la vista de datos v_repos_for_ai

Se confirma la disponibilidad en PostgreSQL de la vista unificada v_repos_for_ai, alineada con las especificaciones del contrato de datos para el módulo de Inteligencia Artificial.

Especificaciones de la Vista
Objeto de acceso: v_repos_for_ai (acceso exclusivo de solo lectura).

Clave primaria de cruce: github_id (BIGINT).

Campos disponibles: github_id, full_name, name, description, homepage, language, topics (TEXT[]), stargazers_count, forks_count, pushed_at, archived, license_spdx_id, content_hash, readme_text.

Reglas de Negocio Implementadas en la Ingesta
Control de Cambios (content_hash): Generación de hash SHA-256 combinando readme_text y pushed_at para permitir el procesamiento incremental y evitar reprocesamientos costosos con LLMs o modelos de embeddings.

Normalización de homepage: Conversión automática de cadenas vacías a valores NULL estrictos.

Exclusión de Repositorios Derivados: Filtrado desde la ingesta de repositorios marcados como fork = True.

Consulta SQL de Referencia para el Módulo de IA

In [ ]:
SELECT 
    github_id, 
    full_name, 
    topics, 
    content_hash, 
    readme_text 
FROM v_repos_for_ai 
WHERE archived = FALSE 
  AND license_spdx_id IS NOT NULL;